# Experiment [5] Visual Dashboard: External Baselines and LEM Brazil Diagnostic

Compare JEPA strict-heldout results to Presto and OlmoEarth frozen embeddings, then inspect why LEM Brazil is the hard failure case.

In [1]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts" / "[5]"
DATA = ROOT / "data"

PALETTE = {
    "surf_jepa_v0": "#2563eb",
    "raw_flattened": "#64748b",
    "raw_stats": "#f97316",
    "surf_jepa_v0_plus_raw_stats": "#16a34a",
    "large_dual_s2": "#2563eb",
    "large_default": "#7c3aed",
    "large_dual_s2_jepa": "#2563eb",
    "large_default_jepa": "#7c3aed",
    "presto": "#db2777",
    "olmoearth": "#059669",
}
PRIORITY_CONDITIONS = ["clean", "sensor_off_s2", "temporal_drop_50", "temporal_drop_70", "s2_off_tdrop50"]
HOLDOUT_ORDER = ["rwanda-ceo", "togo", "togo-eval", "ethiopia", "lem-brazil"]

pd.options.display.max_columns = 80
pd.options.display.max_rows = 80


def standardize_table(df):
    if df.empty:
        return df
    rename = {
        "balanced_acc": "balanced_accuracy",
        "bal_acc": "balanced_accuracy",
        "mean_auc": "auc",
        "mean_f1": "f1",
        "budget": "label_budget",
    }
    return df.rename(columns={k: v for k, v in rename.items() if k in df.columns and v not in df.columns})


def read_csv(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT) if path.is_absolute() and ROOT in path.parents else path}")
        return pd.DataFrame()
    return standardize_table(pd.read_csv(path, **kwargs))


def read_json(path):
    path = Path(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text())


def pct(x):
    return f"{100*x:.1f}%" if pd.notna(x) else ""


def scorecard(df, cols, title="Scorecard", by=None):
    if df.empty:
        print("No data for scorecard")
        return
    work = df.copy()
    if by:
        work = work.set_index(by)
    display(work[cols].style.format({c: "{:.4f}" for c in cols if pd.api.types.is_numeric_dtype(work[c])}).set_caption(title))


def bar_metric(df, x, y, color=None, title=None, barmode="group", text=True, height=460):
    if df.empty:
        print("No data to plot")
        return None
    fig = px.bar(
        df,
        x=x,
        y=y,
        color=color,
        barmode=barmode,
        text_auto=".3f" if text else False,
        color_discrete_map=PALETTE,
        height=height,
        title=title,
    )
    fig.update_layout(template="plotly_white", legend_title_text="", xaxis_title="", yaxis_title=y)
    fig.show()
    return fig


def line_metric(df, x, y, color=None, facet_col=None, title=None, markers=True, height=520):
    if df.empty:
        print("No data to plot")
        return None
    fig = px.line(
        df,
        x=x,
        y=y,
        color=color,
        facet_col=facet_col,
        markers=markers,
        color_discrete_map=PALETTE,
        height=height,
        title=title,
    )
    fig.update_layout(template="plotly_white", legend_title_text="")
    fig.show()
    return fig


def heatmap(df, x, y, z, title=None, height=520, color_continuous_scale="RdYlGn"):
    if df.empty:
        print("No data to plot")
        return None
    pivot = df.pivot_table(index=y, columns=x, values=z, aggfunc="mean")
    fig = px.imshow(
        pivot,
        text_auto=".3f",
        aspect="auto",
        color_continuous_scale=color_continuous_scale,
        height=height,
        title=title,
    )
    fig.update_layout(template="plotly_white", xaxis_title=x, yaxis_title=y)
    fig.show()
    return fig


def priority_average(df, group_cols, metric_cols=("f1", "auc", "balanced_accuracy"), conditions=PRIORITY_CONDITIONS):
    if df.empty:
        return df
    work = df[df["condition"].isin(conditions)].copy() if "condition" in df else df.copy()
    work = work.rename(columns={"balanced_acc": "balanced_accuracy", "mean_auc": "auc", "mean_f1": "f1"})
    available = [c for c in metric_cols if c in work.columns]
    if not available:
        return work.groupby(group_cols, dropna=False).size().reset_index(name="n")
    return work.groupby(group_cols, dropna=False)[available].mean().reset_index()


def normalize_model_name(name):
    text = str(name)
    return text.replace("_seed42", "").replace("_seed7", "").replace("_seed11", "")

RUN_DIR = ARTIFACTS / "external_strict_baselines"
ANALYSIS = RUN_DIR / "analysis"

In [2]:
priority = read_csv(ANALYSIS / "external_vs_jepa_priority.csv")
scorecard(priority, [c for c in priority.columns if c != "baseline"], "[5] External baseline priority", by="baseline")
if not priority.empty:
    long = priority.melt(id_vars="baseline", value_vars=[c for c in ["priority_f1", "priority_auc", "priority_balanced_accuracy"] if c in priority], var_name="metric", value_name="value")
    bar_metric(long, x="metric", y="value", color="baseline", title="[5] External baselines vs JEPA")

,priority_f1,priority_auc,priority_balanced_accuracy,n_rows
baseline,,,,
olmoearth,0.6006,0.7355,0.6778,125.0000
large_dual_s2_jepa,0.4941,0.6703,0.6093,15.0000
large_default_jepa,0.4898,0.6714,0.6088,15.0000
presto,0.3897,0.7289,0.6050,125.0000
raw_flattened,0.3112,0.6111,0.5437,15.0000


In [3]:
holdout = read_csv(ANALYSIS / "external_vs_jepa_holdout_priority.csv")
if not holdout.empty:
    metric = "priority_f1" if "priority_f1" in holdout else "f1"
    heatmap(holdout, x="holdout", y="baseline", z=metric, title="[5] Priority F1 by heldout", height=560)
    metric_auc = "priority_auc" if "priority_auc" in holdout else "auc"
    heatmap(holdout, x="holdout", y="baseline", z=metric_auc, title="[5] Priority AUROC by heldout", height=560, color_continuous_scale="Viridis")

In [4]:
cond = read_csv(ANALYSIS / "external_vs_jepa_condition_summary.csv")
if not cond.empty:
    metric = "f1" if "f1" in cond else "mean_f1"
    heatmap(cond, x="condition", y="baseline", z=metric, title="[5] Stress robustness by external baseline", height=560)

In [5]:
lem = read_csv(ANALYSIS / "lem_top_raw_separability.csv")
if not lem.empty:
    display(lem.head(20))
    ycol = "auc" if "auc" in lem else [c for c in lem.columns if "auc" in c.lower()][0]
    xcol = "feature" if "feature" in lem else lem.columns[0]
    bar_metric(lem.head(15).sort_values(ycol), x=xcol, y=ycol, title="[5] LEM Brazil raw cue separability", height=560)

,band,stat,lem_auc_abs,source_auc_abs,lem_crop_mean,lem_non_crop_mean,source_crop_mean,source_non_crop_mean
0,B8A,mean,0.822094,0.510268,3579.762451,2962.806396,3157.349609,3153.977051
1,B8,mean,0.817003,0.503245,3071.715576,2542.348877,2814.761475,2798.612549
2,B7,mean,0.815008,0.506365,3186.907471,2635.970215,2918.454102,2922.478516
3,B9,mean,0.793575,0.570960,719.799866,616.298584,968.831238,964.348389
4,B8,min,0.786215,0.579901,2341.730225,1914.964478,1534.958984,1676.877563
5,B7,min,0.785321,0.584583,2397.723633,1973.753052,1600.153809,1758.435181
6,B8A,min,0.783235,0.586589,2729.092041,2239.567871,1743.718018,1923.863403
7,B6,mean,0.768762,0.506788,2586.094238,2225.396484,2530.987793,2559.349365
8,B6,min,0.765366,0.572240,2004.059204,1687.266968,1416.341064,1529.393555
9,B9,min,0.761985,0.510265,477.868408,402.484558,433.961731,441.289398


In [6]:
# Raw external probe rows for deeper checks.
frames = []
for baseline in ["presto", "olmoearth"]:
    df = read_csv(RUN_DIR / baseline / "probe_results.csv")
    if not df.empty:
        df["baseline"] = baseline
        frames.append(df)
external_probe = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(external_probe.shape)
external_probe.head()

(250, 11)


,model,condition,label_budget,n_train_sub,n_test,f1,auc,balanced_accuracy,baseline,holdout,seed
0,presto,clean,0.01,641,3591,0.326957,0.708364,0.572651,presto,rwanda-ceo,42
1,presto,clean,0.05,3205,3591,0.153305,0.747021,0.531885,presto,rwanda-ceo,42
2,presto,clean,0.10,6410,3591,0.174442,0.743055,0.531885,presto,rwanda-ceo,42
3,presto,clean,0.25,16025,3591,0.114244,0.739580,0.521911,presto,rwanda-ceo,42
4,presto,clean,1.00,64101,3591,0.173497,0.737571,0.533880,presto,rwanda-ceo,42


In [7]:
if not external_probe.empty:
    curves = external_probe[external_probe["condition"].isin(PRIORITY_CONDITIONS)].groupby(["baseline", "holdout", "label_budget"])[["f1", "auc"]].mean().reset_index()
    line_metric(curves, x="label_budget", y="f1", color="baseline", facet_col="holdout", title="[5] External baseline label-efficiency", height=740)